In [1]:
#Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

In [2]:
#Load dataset
df = pd.read_csv('train.txt',sep = ';',header = None,names = ['text','emotion'])

In [3]:
print(df.head())

                                                text  emotion
0                            i didnt feel humiliated  sadness
1  i can go from feeling so hopeless to so damned...  sadness
2   im grabbing a minute to post i feel greedy wrong    anger
3  i am ever feeling nostalgic about the fireplac...     love
4                               i am feeling grouchy    anger


In [4]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [5]:
print(df['emotion'].unique())
print(df['emotion'].isnull().sum())
print(df.shape)

['sadness' 'anger' 'love' 'surprise' 'fear' 'joy']
0
(16000, 2)


In [6]:
#Label Encoding
emotion_map = {
    "anger": 0,
    "fear": 1,
    "joy": 2,
    "love": 3,
    "sadness": 4,
    "surprise": 5
}

df["emotion"] = df["emotion"].map(emotion_map)

In [7]:
#Preprocessing
df['text'] = df['text'].astype(str).str.lower()
df["emotion"] = df["emotion"].astype(int)

In [8]:
df = df.dropna()

In [9]:
import string

def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))


In [10]:
df['text'] = df['text'].apply(remove_punc)

In [11]:
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new

df['text'] = df['text'].apply(remove_numbers)

In [12]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(remove_emojis)

In [13]:
import nltk

In [14]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [15]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Kush\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Kush\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [16]:
stop_words = set(stopwords.words('english'))


In [17]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [18]:
#Features and target
X = df['text']
Y = df['emotion']

In [19]:
#Train Test Split

X = df['text']

Y = df['emotion']

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42,
    stratify=Y
)

In [20]:
# TF-IDF
tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 3),
    min_df=1,
    max_df=0.9,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Model
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(
    max_iter = 5000,
    C = 10,
    class_weight = 'balanced',
    solver = 'liblinear'
)

# Train
model.fit(X_train_tfidf, Y_train)

# Predict
pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(Y_test, pred))
print(classification_report(Y_test, pred))

c:\Users\Kush\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


Accuracy: 0.875
              precision    recall  f1-score   support

           0       0.88      0.84      0.86       432
           1       0.86      0.83      0.85       387
           2       0.88      0.91      0.89      1072
           3       0.75      0.80      0.77       261
           4       0.93      0.90      0.91       933
           5       0.78      0.78      0.78       115

    accuracy                           0.88      3200
   macro avg       0.85      0.84      0.84      3200
weighted avg       0.88      0.88      0.88      3200



In [21]:
#Testing
test_sentences = [
    "I am feeling so happy today",
    "I am extremely angry with you",
    "I love spending time with you",
    "I feel scared and nervous",
    "I miss my best friend",
    "I am shocked by this news"
]

emotion_reverse = {
    0: "ANGER 😡",
    1: "FEAR 😨",
    2: "JOY 😊",
    3: "LOVE ❤️",
    4: "SADNESS 😢",
    5: "SURPRISE 😲"
}

for text in test_sentences:
    transformed = tfidf.transform([text.lower()])
    prediction = model.predict(transformed)[0]
    print(text, "->", emotion_reverse[prediction])

I am feeling so happy today -> JOY 😊
I am extremely angry with you -> ANGER 😡
I love spending time with you -> LOVE ❤️
I feel scared and nervous -> FEAR 😨
I miss my best friend -> SADNESS 😢
I am shocked by this news -> SURPRISE 😲


In [22]:
# SAVE
import pickle
pickle.dump(model, open("emotion_model.pkl", "wb"))
pickle.dump(tfidf, open("tfidf_vectorizer.pkl", "wb"))